Language: **English (default)** | [Polski](lab.ipynb)

# Lab 10 -- a small Transformer for order-dependent sequence classification

Companion material to [`README.md`](README.md). A different, harder problem
than the lecture: variable-length sequences (real padding) with random
distractors.

**Required core: sections 0-10 (about 100 min + 20 min buffer = 120 min).**
Sections R1-R4 at the end of the notebook are **optional extensions** --
do them only if time remains after finishing the core.

In [ ]:
import time

import torch
import torch.nn as nn

torch.manual_seed(0)
torch.set_num_threads(4)
device = torch.device("cpu")
print(f"Using device: {device}")

## 1. Data: the task generator (given + a short task)

Every sequence (random length from $[10,40]$) contains exactly two
distinguished tokens `A`, `B` at random positions, the rest filled with
random distractors. Label = whether `A` appears before `B`.

In [ ]:
N_DISTRACTORS = 10
TOKEN_A = N_DISTRACTORS
TOKEN_B = N_DISTRACTORS + 1
PAD_ID = N_DISTRACTORS + 2
VOCAB_SIZE = N_DISTRACTORS + 3
MIN_LEN, MAX_LEN = 10, 40


def make_dataset(n, seed):
    g = torch.Generator().manual_seed(seed)
    lengths = torch.randint(MIN_LEN, MAX_LEN + 1, (n,), generator=g)
    seqs = torch.full((n, MAX_LEN), PAD_ID, dtype=torch.long)
    labels = torch.zeros(n, dtype=torch.long)
    for i in range(n):
        length = lengths[i].item()
        distractors = torch.randint(0, N_DISTRACTORS, (length,), generator=g)
        pos_ab = torch.randperm(length, generator=g)[:2]
        pa, pb = pos_ab[0].item(), pos_ab[1].item()
        distractors[pa] = TOKEN_A
        distractors[pb] = TOKEN_B
        seqs[i, :length] = distractors
        labels[i] = 1 if pa < pb else 0
    return seqs, lengths, labels


X_train, len_train, y_train = make_dataset(1500, seed=0)
X_val, len_val, y_val = make_dataset(300, seed=1)
X_test, len_test, y_test = make_dataset(300, seed=2)
print(f"train: {X_train.shape[0]} examples, X.shape={tuple(X_train.shape)}")
print(f"example sequence (length {len_train[0].item()}):")
print(X_train[0, :len_train[0]].tolist())
print(f"label: {y_train[0].item()}")

**Task:** verify class balance and the independence of sequence length from
the label.

In [ ]:
# TODO:
# 1. print the class balance (y_train.float().mean())
# 2. compute mean sequence length SEPARATELY for each class (len_train[y_train==0]
#    vs len_train[y_train==1]) - they should be close
raise NotImplementedError("TODO: balance and length-independence verification")

## 2. A baseline without order -- your task

Build a classifier based on averaging token embeddings (no attention) --
it **must skip padding** when averaging. You are given a helper
`masked_mean_pool` (exactly the same principle as in module 9, section 6 --
already mastered, we do not reimplement it from scratch) -- you decide
**where and how** to use it.

In [ ]:
def masked_mean_pool(x, real_mask, lengths):
    """Averages `x` (B,T,D) ONLY over positions where `real_mask` (B,T, bool)
    is True, dividing by `lengths` (not by T) - the same principle as in
    module 9, section 6 (pooling that excludes padding)."""
    real_mask = real_mask.unsqueeze(-1).float()
    return (x * real_mask).sum(dim=1) / lengths.unsqueeze(1).float()


D_MODEL = 16


class MeanPoolBaseline(nn.Module):
    def __init__(self, vocab_size, d_model=D_MODEL, n_classes=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.fc = nn.Linear(d_model, n_classes)

    def forward(self, x, lengths):
        # TODO:
        # 1. e = self.embed(x)  (B, T, d_model)
        # 2. real_mask = (x != PAD_ID)  (B, T) bool - True = real token
        # 3. pooled = masked_mean_pool(e, real_mask, lengths)
        # 4. return self.fc(pooled)
        raise NotImplementedError("TODO: MeanPoolBaseline.forward")

## 3. Positional encoding (given)

In [ ]:
def sinusoidal_positional_encoding(seq_len, d_model):
    position = torch.arange(seq_len).unsqueeze(1).float()
    div_term = torch.exp(
        torch.arange(0, d_model, 2).float() * (-torch.log(torch.tensor(10000.0)) / d_model)
    )
    pe = torch.zeros(seq_len, d_model)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

## 4. Padding mask -- your task

Build a `key_padding_mask` from `x` and `PAD_ID`. **Critical:** you are using
`nn.MultiheadAttention`, so `key_padding_mask` must use the convention
**`True` = this is padding, ignore it** -- **not** `True` = allowed (this
convention is **opposite** to `F.scaled_dot_product_attention`). Mix up
these conventions and you will land squarely in the diagnostic task in
section 9.

In [ ]:
def make_key_padding_mask(x):
    # TODO: return a bool tensor (B, T) - True WHERE x == PAD_ID (padding,
    # to be ignored), following the nn.MultiheadAttention convention
    raise NotImplementedError("TODO: make_key_padding_mask")


demo_mask = make_key_padding_mask(X_train[0:1])  # noqa: F821 - defined above
print(f"mask for a sequence of length {len_train[0].item()} (out of {MAX_LEN} positions):")
print(demo_mask)
print(f"number of True (padding) in the mask: {demo_mask.sum().item()} "
      f"(should be {MAX_LEN - len_train[0].item()})")
assert demo_mask.sum().item() == MAX_LEN - len_train[0].item()

## 5. Transformer block -- your task

Build `TinyTransformerBlock`: self-attention (`nn.MultiheadAttention` -- the
Q/K/V projections do not need to be written by hand, this is already
verified in the lecture) **+ residual connection + LayerNorm + feed-forward
(Linear->ReLU->Linear) + a second residual connection + a second LayerNorm**
(Post-LN, as in `lecture.md`). **You** combine these components -- do not
hide the architecture behind a single call to `nn.TransformerEncoderLayer`.

In [ ]:
class TinyTransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.mha = nn.MultiheadAttention(d_model, n_heads, batch_first=True, dropout=0.0)
        self.ln1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x, key_padding_mask):
        # TODO:
        # 1. attn_out, attn_weights = self.mha(x, x, x,
        #    key_padding_mask=key_padding_mask, need_weights=True,
        #    average_attn_weights=False)
        # 2. x = self.ln1(x + attn_out)  (residual connection + LayerNorm)
        # 3. x = self.ln2(x + self.ff(x))  (second residual connection + LayerNorm)
        # 4. return (x, attn_weights)
        raise NotImplementedError("TODO: TinyTransformerBlock.forward")

## 6. Aggregation and classifier -- your task

Build `Classifier`: `nn.Embedding` (+ positional encoding from section 3, if
`use_pos_enc=True`) -> the `TinyTransformerBlock` from section 5 ->
**aggregation to a single vector per sequence** (use `masked_mean_pool` from
section 2 -- the same principle: exclude padding, divide by `lengths`) -> a
classification layer.

In [ ]:
N_HEADS, D_FF = 2, 32


class Classifier(nn.Module):
    def __init__(self, vocab_size, d_model=D_MODEL, n_heads=N_HEADS, d_ff=D_FF,
                 use_pos_enc=True, n_classes=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.use_pos_enc = use_pos_enc
        if use_pos_enc:
            self.register_buffer("pos_enc", sinusoidal_positional_encoding(MAX_LEN, d_model))
        self.block = TinyTransformerBlock(d_model, n_heads, d_ff)
        self.fc = nn.Linear(d_model, n_classes)

    def forward(self, x, lengths):
        # TODO:
        # 1. key_padding_mask = make_key_padding_mask(x)
        # 2. e = self.embed(x); if self.use_pos_enc, add self.pos_enc
        # 3. out, attn_weights = self.block(e, key_padding_mask)
        # 4. pooled = masked_mean_pool(out, ~key_padding_mask, lengths)  (the same
        #    function as in section 2 - real_mask is the NEGATION of key_padding_mask)
        # 5. return (self.fc(pooled), attn_weights)
        raise NotImplementedError("TODO: Classifier.forward")

**Verification (a ready-made assertion): the same sequence must give an
identical representation regardless of what padding it ends up with in a
batch.**

In [ ]:
torch.manual_seed(0)
check_model = Classifier(VOCAB_SIZE)
check_model.eval()

short_idx = (len_train < MAX_LEN).nonzero()[0, 0].item()
short_seq, short_len = X_train[short_idx:short_idx + 1], len_train[short_idx:short_idx + 1]
longer_idx = len_train.argmax().item()
longer_seq, longer_len = X_train[longer_idx:longer_idx + 1], len_train[longer_idx:longer_idx + 1]

batched_X = torch.cat([short_seq, longer_seq], dim=0)
batched_len = torch.cat([short_len, longer_len], dim=0)

with torch.no_grad():
    logits_alone, _ = check_model(short_seq, short_len)
    logits_batched, _ = check_model(batched_X, batched_len)

match = torch.allclose(logits_alone, logits_batched[0:1], atol=1e-5)
print(f"same sequence, same result regardless of batch padding: {match}")
assert match, (
    'BUG: the representation depends on padding - '
    'check the mask (section 4) and aggregation (6)'
)

## 7. Training (given) and comparing with vs. without positional encoding -- your task

In [ ]:
def train_eval(model, epochs=15, lr=0.003, batch_size=32, seed=0):
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    n = X_train.shape[0]
    start = time.time()
    for _ in range(epochs):
        perm = torch.randperm(n)
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            opt.zero_grad()
            logits, _ = model(X_train[idx], len_train[idx])
            loss = loss_fn(logits, y_train[idx])
            loss.backward()
            opt.step()
    train_time = time.time() - start
    model.eval()
    with torch.no_grad():
        val_logits, _ = model(X_val, len_val)
        val_acc = (val_logits.argmax(1) == y_val).float().mean().item()
        test_logits, _ = model(X_test, len_test)
        test_acc = (test_logits.argmax(1) == y_test).float().mean().item()
    return val_acc, test_acc, train_time

In [ ]:
baseline_model = MeanPoolBaseline(VOCAB_SIZE)  # noqa: F821 - defined in section 2
torch.manual_seed(0)
opt_b = torch.optim.Adam(baseline_model.parameters(), lr=0.003)
loss_fn_b = nn.CrossEntropyLoss()
for _ in range(15):
    perm_b = torch.randperm(X_train.shape[0])
    for i in range(0, X_train.shape[0], 32):
        idx = perm_b[i:i + 32]
        opt_b.zero_grad()
        loss_b = loss_fn_b(baseline_model(X_train[idx], len_train[idx]), y_train[idx])
        loss_b.backward()
        opt_b.step()
baseline_model.eval()
with torch.no_grad():
    baseline_val_acc = (baseline_model(X_val, len_val).argmax(1) == y_val).float().mean().item()
print(f"baseline (mean-pool, no order): val_acc={baseline_val_acc:.3f}")

**Task:** train `Classifier` with `use_pos_enc=True` **and** with
`use_pos_enc=False` (the same training budget). Save the models as
`model_with_pe`/`model_without_pe` -- `model_with_pe` is reused in section 8
and in extensions R1/R2, **do not train it a second time**.

In [ ]:
# TODO:
# 1. model_with_pe = Classifier(VOCAB_SIZE, use_pos_enc=True); train_eval(model_with_pe)
# 2. model_without_pe = Classifier(VOCAB_SIZE, use_pos_enc=False); train_eval(model_without_pe)
# 3. print val_acc and test_acc for both
raise NotImplementedError("TODO: with/without PE comparison")

**Multi-seed check (given).** Training takes mere seconds on CPU, so the
loop below runs it 6 times (3 seeds x 2 variants) -- **your task is
interpreting** the result in section 10 (Conclusions, question 3), not
writing this loop: is the "without PE = chance level" result stable, or is
it a fluke of a single unlucky run?

In [ ]:
print("multi-seed check (3 seeds):")
for seed in [0, 1, 2]:
    m1 = Classifier(VOCAB_SIZE, use_pos_enc=True)
    v1, t1, _ = train_eval(m1, seed=seed)
    m2 = Classifier(VOCAB_SIZE, use_pos_enc=False)
    v2, t2, _ = train_eval(m2, seed=seed)
    print(f"seed={seed}: with_PE val={v1:.3f} test={t1:.3f} | "
          f"without_PE val={v2:.3f} test={t2:.3f}")

## 8. Evaluation on the test set -- your task

Evaluate `model_with_pe` from section 7 (**do not retrain it**): accuracy on
the test set + the length distribution of misclassified examples (do errors
concentrate at particular lengths?).

In [ ]:
# TODO:
# 1. compute model_with_pe's predictions ON X_test (without retraining)
# 2. test_acc = accuracy on X_test
# 3. find the indices of misclassified examples and their lengths - do they
#    concentrate at particular lengths?
raise NotImplementedError("TODO: test evaluation")

## 9. Diagnostic task

The code below (`BuggyClassifier`) **looks** correct -- it defines a mask,
builds the block, trains without raising an exception -- but has a bug
related to the mask convention, exactly the one section 4 warns about.

In [ ]:
class BuggyClassifier(nn.Module):
    def __init__(self, vocab_size, d_model=D_MODEL, n_heads=N_HEADS, d_ff=D_FF):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.register_buffer("pos_enc", sinusoidal_positional_encoding(MAX_LEN, d_model))
        self.block = TinyTransformerBlock(d_model, n_heads, d_ff)
        self.fc = nn.Linear(d_model, 2)

    def forward(self, x, lengths):
        key_padding_mask = x != PAD_ID  # BUG: True = "this is a real token"
        e = self.embed(x) + self.pos_enc.unsqueeze(0)
        out, _ = self.block(e, key_padding_mask)
        real_mask = (x != PAD_ID).unsqueeze(-1).float()
        pooled = (out * real_mask).sum(dim=1) / lengths.unsqueeze(1).float()
        return self.fc(pooled)


torch.manual_seed(0)
buggy_model = BuggyClassifier(VOCAB_SIZE)
opt_bug = torch.optim.Adam(buggy_model.parameters(), lr=0.003)
loss_fn_bug = nn.CrossEntropyLoss()

print("training BuggyClassifier - watch the loss after each epoch:")
for epoch in range(5):
    perm_bug = torch.randperm(X_train.shape[0])
    total_loss, n_batches = 0.0, 0
    for i in range(0, X_train.shape[0], 32):
        idx = perm_bug[i:i + 32]
        opt_bug.zero_grad()
        logits = buggy_model(X_train[idx], len_train[idx])
        loss = loss_fn_bug(logits, y_train[idx])
        loss.backward()
        opt_bug.step()
        total_loss += loss.item()
        n_batches += 1
    print(f"  epoch {epoch}: loss = {total_loss / n_batches}")

**Task:**
1. Diagnose the bug -- which line, and why.
2. Run training and observe what happens to the loss (hint: check whether the
   training set contains sequences **with no padding at all**, i.e. of
   length `MAX_LEN` -- and what happens to attention masking when **every**
   position of such a sequence ends up "blocked" by the inverted
   convention).
3. Explain mathematically, referring to the "Rows with all keys masked"
   section in `lecture.md`: what exactly happens to `softmax` when, for a
   given query, zero keys are allowed?
4. Fix the code (invert the mask convention).
5. Verify that after the fix the loss is always finite (`torch.isfinite`)
   and that the model trains to an accuracy close to the original, correct
   implementation.

In [ ]:
# TODO:
# 1. as a comment: diagnose exactly which line is the source of the bug and why
# 2. check: are there sequences of length MAX_LEN in train (len_train == MAX_LEN)?
# 3. write FixedClassifier - a corrected version of BuggyClassifier
# 4. train FixedClassifier, checking torch.isfinite(loss) at EVERY step
# 5. compare the final val_acc with the original, correct Classifier
raise NotImplementedError("TODO: diagnosis and fix")

## 10. Conclusions -- your task

**1. Which architectural elements (sections 5-6) are responsible for: (a) mixing information between positions, (b) introducing positional information, (c) reducing the sequence to a single vector?**

_(Your answer here.)_

**2. How large is the accuracy gap between the baseline (section 2) and the Transformer with PE (section 7)?**

_(Your answer here.)_

**3. Is the "without PE = chance level" result stable across several seeds?**

_(Your answer here.)_

**4. Is the masking correct -- do you have proof that the model does not use information contained in the padding?**

_(Your answer here.)_

**5. What exactly does (and does not) this experiment show about the attention mechanism?**

_(Your answer here.)_

**6. What is at least one limitation of this experiment?**

_(Your answer here.)_

---

## Extensions (optional, if time remains after sections 0-10)

The sections below are **not** part of the required core. Do them only if
time remains -- they reuse `model_with_pe` from section 7, without
retraining it.

### R1. Behavior across different sequence lengths

For `model_with_pe`: compute validation-set accuracy in at least 3 length
buckets (e.g. $[10,19]$, $[20,29]$, $[30,40]$). Is accuracy stable, or does
it drop for longer/shorter sequences?

In [ ]:
# TODO:
# 1. compute model_with_pe's predictions on X_val (without retraining)
# 2. for at least 3 length buckets (e.g. [10,19],[20,29],[30,40]) compute
#    accuracy ONLY on examples with len_val in that bucket
raise NotImplementedError("TODO (extension): accuracy by length")

### R2. Analysis of selected attention weights

For one correctly classified test example: print the sequence, the
positions of tokens `A` and `B`, and the attention weights (query = position
of token `A`, any head). **Do not interpret** a high weight toward `B` as
"proof" that the model "understands" the order relation -- only describe
what you actually see in the numbers.

In [ ]:
# TODO:
# 1. pick one correctly classified test example (use model_with_pe,
#    without retraining)
# 2. find the positions of tokens A and B in this sequence
# 3. print the attention weights (any head) for query = position of A, ONLY
#    for the real (non-padding) positions
raise NotImplementedError("TODO (extension): attention weight analysis")

**Task (Markdown):** is a high weight toward a specific position "proof"
here that the model "understands" the order relation? Refer to the
corresponding "❌/✅" box from `lecture.md`.

_(Your answer here.)_

### R3/R4. Architecture

**R3.** Stack **two** `TinyTransformerBlock`s one after another -- does this
improve accuracy or stability on longer sequences?

**R4.** Swap Post-LN for Pre-LN (norm before the sublayer, not after) and
check whether anything changes in training stability.